In [2]:
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/nabill/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/nabill/tugas4/
print("Berhasil diunggah ke HDFS: /user/nabill/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/nabill/tugas4/transaksi_september_2026.csv


In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("TugasMandiriPertemuan4")\
    .master("local[*]")\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

path_hdfs="hdfs://localhost:9000/user/nabill/tugas4/transaksi_september_2026.csv"
df=spark.read.csv(path_hdfs, header=True, inferSchema=True)

df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

26/09/10 17:02:58 WARN Utils: Your hostname, nabill-IdeaPad-3-14IML05 resolves to a loopback address: 127.0.1.1; using 10.90.70.70 instead (on interface wlp0s20f3)
26/09/10 17:02:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 17:02:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [9]:
from pyspark.sql.functions import col

jumlah_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah nilai kosong pada kolom rating:", jumlah_kosong)

df_clean = df.na.fill({"rating": 3.0})

print("Jumlah nilai kosong setelah ditangani:", df_clean.filter(col("rating").isNull()).count())

Jumlah nilai kosong pada kolom rating: 204
Jumlah nilai kosong setelah ditangani: 0


Alasan penanganan data kosong : Metode 'na.fill()' digunakan untuk mengisikan angka netral (3.0) pada kolom rating yang kosong agar seluruh baris transaksi tetap dipertahankan, kalau menggunakan 'na.drop()', sebanyak 204 baris transaksi akan hilang dari dataset, sehingga analisis total pendapatan pada tahap selanjutnya menjadi tidak akurat

In [11]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

df_clean = df_clean.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df_clean.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan","tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [18]:
from pyspark.sql.functions import sum as spark_sum, count, avg

print('===1. Kategori dengan total pendapatan tertinggi ====')
df_clean.groupBy("kategori")\
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan"))\
    .orderBy(col("total_pendapatan").desc())\
    .show(1)
print('===2. Kota dengan transaksi tier "Besar" terbanyak===')
df_clean.filter(col("tier_transaksi") == "Besar")\
    .groupBy("kota")\
    .agg(count("order_id").alias("jumlah_transaksi_besar"))\
    .orderBy(col("jumlah_transaksi_besar").desc())\
    .show(1)
print('===3. Rata rata rating per metode pembayaran====')
df_clean.groupBy("metode_pembayaran")\
    .agg(avg("rating").alias("rata_rata_rating"))\
    .show()

===1. Kategori dengan total pendapatan tertinggi ====


+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

===2. Kota dengan transaksi tier "Besar" terbanyak===


+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

===3. Rata rata rating per metode pembayaran====


+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|3.948207171314741|
|    Transfer Bank|3.932806324110672|
|     Kartu Kredit|3.861788617886179|
|         E-Wallet|            3.904|
+-----------------+-----------------+



In [21]:
path_hdfs_output = "hdfs://localhost:9000/user/nabill/tugas4/hasil_september_2026"

df_clean.write.mode("overwrite").csv(path_hdfs_output, header=True)

!hdfs dfs -ls /user/nabill/tugas4/hasil_september_2026

Found 2 items
-rw-r--r--   3 nabill supergroup          0 2026-09-10 17:48 /user/nabill/tugas4/hasil_september_2026/_SUCCESS
-rw-r--r--   3 nabill supergroup      97296 2026-09-10 17:48 /user/nabill/tugas4/hasil_september_2026/part-00000-6a22bda2-7825-4f92-9ba8-a38f75393d1f-c000.csv


spark menghasilkan output berupa beberapa berkas partisi ('part-00000...', 'part-00001...') karena Spark beroperasi dalam arsitektur komputasi terdistribusi secara paralel. setiap task pemrosesan menuliskan porsi data dari memorinya masing-masing ke HDFS secara bersamaan tanpa saling menunggu. Mekanisme ini mencegah kemacetan (bottleneck) saat menulis data berukuran sangat besar.

In [4]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder \
        .appName()\
        .getOrCreate()

TypeError: SparkSession.Builder.appName() missing 1 required positional argument: 'name'